# Notebook to calculate final statistics for each dataset used

stats to use:
avg doc length
avg sentence length
avg named entity length

In [3]:
## get all paths first
indian_train = r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\inlegal_train_FINAL.conll"
indian_val = r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\inlegal_validation_FINAL.conll"
edgar_train = r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\EDGAR_TRAINING.conll"
edgar_val = r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\EDGAR_VALIDATION.conll"
opensource_train = r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\OS_TRAINING.conll"
opensource_val = r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\OS_VALIDATION.conll"
llm_train = r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\LLM-eval\LLM_train.conll"
llm_dev = r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\LLM-eval\LLM_dev.conll"
mixed_train = r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\LLM-eval\COMBINED_train.conll"
mixed_dev = r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\LLM-eval\COMBINED_dev.conll"
llm_folder = r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\LLM-eval"
test_file = r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\datasets\my_data\final_dataset_1st_may.conll"

collection_files = [
    "\labeled_ivm.conll", "\labeled_ecjd.conll", "\labeled_et.conll",
    "\labeled_irs.conll", "\labeled_hrte.conll", "\labeled_gttc2.conll",
    "\labeled_ecji.conll", "\labeled_bit.conll", "\labeled_wht.conll",
    "\labeled_tns.conll", "\labeled_ttcls.conll"
]
coll_paths = []


for file in collection_files:
    # name = file[8:-6]
    comb = llm_folder + file
    coll_paths.append(comb)

print(coll_paths)
    

['C:\\Users\\M.Walavalkar\\OneDrive - IBFD\\Desktop\\thesis-ner-manya-explore\\LLM-eval\\labeled_ivm.conll', 'C:\\Users\\M.Walavalkar\\OneDrive - IBFD\\Desktop\\thesis-ner-manya-explore\\LLM-eval\\labeled_ecjd.conll', 'C:\\Users\\M.Walavalkar\\OneDrive - IBFD\\Desktop\\thesis-ner-manya-explore\\LLM-eval\\labeled_et.conll', 'C:\\Users\\M.Walavalkar\\OneDrive - IBFD\\Desktop\\thesis-ner-manya-explore\\LLM-eval\\labeled_irs.conll', 'C:\\Users\\M.Walavalkar\\OneDrive - IBFD\\Desktop\\thesis-ner-manya-explore\\LLM-eval\\labeled_hrte.conll', 'C:\\Users\\M.Walavalkar\\OneDrive - IBFD\\Desktop\\thesis-ner-manya-explore\\LLM-eval\\labeled_gttc2.conll', 'C:\\Users\\M.Walavalkar\\OneDrive - IBFD\\Desktop\\thesis-ner-manya-explore\\LLM-eval\\labeled_ecji.conll', 'C:\\Users\\M.Walavalkar\\OneDrive - IBFD\\Desktop\\thesis-ner-manya-explore\\LLM-eval\\labeled_bit.conll', 'C:\\Users\\M.Walavalkar\\OneDrive - IBFD\\Desktop\\thesis-ner-manya-explore\\LLM-eval\\labeled_wht.conll', 'C:\\Users\\M.Walavalka

In [6]:
def read_conll_to_lists(filepath):
    all_tokens = []
    all_labels = []
    current_tokens = []
    current_labels = []

    with open(filepath, "r", encoding="utf-8") as infile:
        for line in infile:
            line = line.rstrip("\n")
            if line.strip() == "":
                if current_tokens:
                    all_tokens.append(current_tokens)
                    all_labels.append(current_labels)
                    current_tokens = []
                    current_labels = []
            else:
                parts = line.split("\t")
                if len(parts) == 2:
                    token, label = parts[0], parts[1]
                    current_tokens.append(token)
                    current_labels.append(label)
        if current_tokens:
            all_tokens.append(current_tokens)
            all_labels.append(current_labels)

    return all_tokens, all_labels

In [17]:
def avg_sentence_length(file):
    """this function gets the avg sent length in tokens"""
    sentences, labels = read_conll_to_lists(file)

    total_sents = len(sentences)
    overall_sent_length = sum(len(sent) for sent in sentences)

    print(f"Total number of sentences: {total_sents}")
    print(f"Overall sentence lengths in tokens: {overall_sent_length}")
    avg_sent_length = overall_sent_length / total_sents if total_sents else 0
    print(f"Average sentence length in tokens: {avg_sent_length}")
    return avg_sent_length


def avg_token_length(file):
    """this function gets the avg token length in characters"""
    sentences, labels = read_conll_to_lists(file)
    total_char_length = 0
    total_tokens = 0

    for tokens in sentences:
        if tokens and tokens[0] == "-DOCSTART-":
            continue
        for token in tokens:
            total_char_length += len(token)
            total_tokens += 1

    print(f"Total tokens: {total_tokens}")
    print(f"Total character length across tokens: {total_char_length}")
    avg = total_char_length / total_tokens if total_tokens else 0
    print(f"Average token length (chars): {avg}")
    return avg

def avg_named_entity_length(file):
    """this function gets the avg named entity length in tokens"""
    sentences, all_labs = read_conll_to_lists(file)
    entity_lengths = []

    for tokens, labels in zip(sentences, all_labs):
        if tokens and tokens[0] == "-DOCSTART-":
            continue
        current = 0
        for label in labels:
            if label.startswith("B-"):
                if current:
                    entity_lengths.append(current)
                current = 1
            elif label.startswith("I-") and current:
                current += 1
            else:
                if current:
                    entity_lengths.append(current)
                current = 0
        if current:
            entity_lengths.append(current)

    total_entities = len(entity_lengths)
    total_tokens = sum(entity_lengths)
    avg = total_tokens / total_entities if total_entities else 0
    print(f"Total NEs: {total_entities}")
    print(f"Total tokens in entities: {total_tokens}")
    print(f"Average named entity length (tokens): {avg}")
    return avg

avg_sentence_length(indian_train)
avg_token_length(indian_train)
avg_named_entity_length(indian_train)


Total number of sentences: 10995
Overall sentence lengths in tokens: 607671
Average sentence length in tokens: 55.26793997271487
Total tokens: 607671
Total character length across tokens: 2494294
Average token length (chars): 4.104678353911903
Total NEs: 20643
Total tokens in entities: 55643
Average named entity length (tokens): 2.69548999660902


2.69548999660902

In [16]:
for file in coll_paths:
    print(file[88:-6])
    avg_token_length_per_doc(file)
    print(file[88:-6])
    avg_sentence_length_per_doc(file)
    print(file[88:-6])
    avg_named_entity_length(file)

ivm
Total docs: 6
  Doc 1: 3186 tokens, avg token length = 4.43 chars
  Doc 2: 932 tokens, avg token length = 4.38 chars
  Doc 3: 2027 tokens, avg token length = 4.36 chars
  Doc 4: 8321 tokens, avg token length = 4.50 chars
  Doc 5: 3039 tokens, avg token length = 4.57 chars
  Doc 6: 2035 tokens, avg token length = 4.37 chars
Avg token length across all docs: 4.44
ivm
Total docs: 6
  Doc 1: 390 sentences, avg sentence length = 8.17 tokens
  Doc 2: 113 sentences, avg sentence length = 8.25 tokens
  Doc 3: 253 sentences, avg sentence length = 8.01 tokens
  Doc 4: 921 sentences, avg sentence length = 9.03 tokens
  Doc 5: 374 sentences, avg sentence length = 8.13 tokens
  Doc 6: 249 sentences, avg sentence length = 8.17 tokens
Average sentence length across all documents: 8.29
ivm
Total NEs: 1600
Total tokens in entities: 3500
Average named entity length (tokens): 2.1875
ecjd
Total docs: 30
  Doc 1: 14434 tokens, avg token length = 4.38 chars
  Doc 2: 7840 tokens, avg token length = 4.58 